# Taller 2 — Notebook 03: Exploración de los datos limpios

**MINE-4101 · Predicción de churn para FinAlpes** · **Integrantes:** Daniel Triviño · Nicolás Bedoya

Distribución de cada variable y su relación con el churn. Los faltantes se excluyen de cada gráfico.

> Ejecutar después de `02_limpieza`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')

RUTA_LIMPIO = '../data/processed/churn_limpio.parquet'
RUTA_FIGS = '../figs'
os.makedirs(RUTA_FIGS, exist_ok=True)

df = pd.read_parquet(RUTA_LIMPIO)
# credit_score y age vienen como enteros que admiten faltantes (Int64); para graficar y calcular estadísticos
# los pasamos a float. Sus valores siguen siendo enteros.
df[['credit_score', 'age']] = df[['credit_score', 'age']].astype(float)

OBJETIVO = 'churn'
tasa_global = df[OBJETIVO].mean() * 100
print(f'Filas: {len(df):,}  |  columnas: {df.shape[1]}  |  tasa de churn: {tasa_global:.2f}%')


In [ ]:
# Grupos para graficar, según cuántos valores distintos tiene cada variable (no según su tipo):
# con muchos valores conviene un histograma; con pocos, barras.
NUM_MUCHOS_VALORES = ['credit_score', 'age', 'balance', 'estimated_salary']
NUM_POCOS_VALORES = ['tenure', 'products_number']
CATEGORICAS = ['country', 'gender']
BINARIAS = ['credit_card', 'active_member']


## 1. Variables numéricas

In [ ]:
numericas = NUM_MUCHOS_VALORES + NUM_POCOS_VALORES
resumen = df[numericas].describe().T[['mean', '50%', 'std', 'min', 'max']]
resumen.columns = ['media', 'mediana', 'desv_est', 'min', 'max']
resumen['asimetria'] = df[numericas].skew()
resumen['pct_ceros'] = (df[numericas] == 0).mean() * 100
resumen['faltantes'] = df[numericas].isna().sum()
resumen


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), NUM_MUCHOS_VALORES):
    sns.histplot(df[col].dropna(), bins=40, ax=ax, color='#4C72B0')
    ax.axvline(df[col].median(), color='#C44E52', ls='--', lw=1.5, label='mediana')
    ax.set_title(col); ax.set_xlabel(''); ax.legend()
fig.suptitle('Distribución de las variables numéricas con muchos valores', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/03_hist_numericas.png', bbox_inches='tight'); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, NUM_POCOS_VALORES):
    conteo = df[col].value_counts().sort_index()
    ax.bar(conteo.index.astype(str), conteo.values, color='#4C72B0')
    ax.set_title(col); ax.set_ylabel('clientes')
fig.suptitle('Distribución de las variables numéricas con pocos valores', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/03_barras_discretas.png', bbox_inches='tight'); plt.show()


**Qué vemos:**

- `credit_score` es casi simétrico (media 651, mediana 652). `estimated_salary` es prácticamente uniforme entre 0 y
  200.000, sin ningún patrón.
- `age` tiene cola a la derecha (asimetría 1,0): la mayoría de los clientes tiene entre 30 y 45 años, y la mediana es 37.
- `balance` tiene dos grupos. El 35,6 % de los clientes tiene saldo 0, y el resto se concentra entre 100.000 y 150.000.
  Por eso la media (76.310) queda por debajo de la mediana (97.041).
- `tenure` es casi uniforme entre 1 y 9 años. En `products_number`, casi todos los clientes tienen 1 o 2 productos.

## 2. Variables categóricas y binarias

In [ ]:
cols = CATEGORICAS + BINARIAS
fig, axes = plt.subplots(1, len(cols), figsize=(16, 4))
for ax, col in zip(axes, cols):
    conteo = df[col].value_counts().sort_index()
    pct = conteo / conteo.sum() * 100
    barras = ax.bar(conteo.index.astype(str), conteo.values, color='#55A868')
    for barra, p in zip(barras, pct):
        ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height(), f'{p:.0f}%', ha='center', va='bottom')
    ax.set_ylim(0, conteo.max() * 1.15)
    ax.set_title(col); ax.set_ylabel('clientes')
fig.suptitle('Distribución de las variables categóricas y binarias', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/03_barras_categoricas.png', bbox_inches='tight'); plt.show()


**Qué vemos:**

- Francia tiene la mitad de los clientes (50 %). Alemania y España tienen un cuarto cada una.
- Hay algo más de hombres (55 %) que de mujeres (45 %).
- El 71 % de los clientes tiene tarjeta de crédito. Activos e inactivos están casi a la par (51 % y 49 %).
- Todos los grupos son lo bastante grandes para estimar su tasa de abandono.

## 3. Cada variable frente al churn

### 3.1 Tasa de churn por grupo

Alto de la barra = clientes del grupo; rojo = los que abandonan; encima, la tasa y el conteo. `saldo_en_cero` es un
grupo exploratorio, no una columna del dataset.

In [ ]:
def tasa_churn(datos, col):
    '''Clientes, abandonos y tasa de churn (%) por cada valor de `col`.'''
    t = datos.groupby(col, observed=True)[OBJETIVO].agg(clientes='size', abandonos='sum', tasa='mean')
    t['tasa'] = t['tasa'] * 100
    return t

explorar = df.assign(saldo_en_cero=np.where(df['balance'].isna(), np.nan, (df['balance'] == 0).astype(float)))
GRUPOS = ['country', 'gender', 'products_number', 'active_member', 'credit_card', 'tenure', 'saldo_en_cero']
tasas = {col: tasa_churn(explorar, col) for col in GRUPOS}

def barras_churn(ax, t, titulo, fuente=8):
    '''Barras apiladas: alto = clientes del grupo, rojo = los que abandonan. Encima: tasa y conteo exacto.'''
    etiquetas = [str(int(i)) if isinstance(i, float) else str(i) for i in t.index]
    se_quedan = t['clientes'] - t['abandonos']
    ax.bar(etiquetas, se_quedan, color='#4C72B0')
    ax.bar(etiquetas, t['abandonos'], bottom=se_quedan, color='#C44E52')
    for x, (n, a, tasa) in enumerate(zip(t['clientes'], t['abandonos'], t['tasa'])):
        ax.text(x, n, f'{tasa:.1f}%\n{a:,} de {n:,}', ha='center', va='bottom', fontsize=fuente)
    ax.set_ylim(0, t['clientes'].max() * 1.28)
    ax.set_title(titulo); ax.set_ylabel('clientes')

fig = plt.figure(figsize=(18, 9))
rejilla = fig.add_gridspec(2, 4)
paneles = {'country': rejilla[0, 0], 'gender': rejilla[0, 1], 'products_number': rejilla[0, 2],
           'active_member': rejilla[0, 3], 'credit_card': rejilla[1, 0], 'saldo_en_cero': rejilla[1, 1],
           'tenure': rejilla[1, 2:]}
for col, panel in paneles.items():
    barras_churn(fig.add_subplot(panel), tasas[col], col, fuente=7 if col == 'tenure' else 8)

leyenda = [plt.Rectangle((0, 0), 1, 1, color='#4C72B0'), plt.Rectangle((0, 0), 1, 1, color='#C44E52')]
fig.legend(leyenda, ['se queda', 'abandona'], loc='upper right', ncol=2)
fig.suptitle(f'Churn por grupo: alto = clientes del grupo; encima, % que abandona y conteo '
             f'(promedio general: {tasa_global:.1f}%)', fontweight='bold')
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f'{RUTA_FIGS}/03_tasa_churn_por_grupo.png', bbox_inches='tight'); plt.show()


In [ ]:
with pd.option_context('display.float_format', lambda x: f'{x:,.1f}'):
    for col, t in tasas.items():
        print(f'--- {col} ---')
        display(t)


**Diferencia de churn entre el grupo más alto y el más bajo** (solo grupos con al menos `MIN_CLIENTES` clientes).

In [ ]:
MIN_CLIENTES = 50

filas = []
for col, t in tasas.items():
    t = t[t['clientes'] >= MIN_CLIENTES]
    filas.append({'variable': col,
                  'grupo_mayor_tasa': t['tasa'].idxmax(), 'mayor_tasa': t['tasa'].max(),
                  'grupo_menor_tasa': t['tasa'].idxmin(), 'menor_tasa': t['tasa'].min(),
                  'diferencia_pp': t['tasa'].max() - t['tasa'].min()})
diferencias = pd.DataFrame(filas).set_index('variable').sort_values('diferencia_pp', ascending=False)
with pd.option_context('display.float_format', lambda x: f'{x:,.1f}'):
    display(diferencias)


**Qué vemos:**

- El número de productos es la variable que más separa. Con 2 productos abandona el 7,6 % y con 1, el 27,7 %. Con 3 o 4
  abandona el 82,7 % y el 100 %, aunque son grupos pequeños (266 y 60 clientes). La relación tiene forma de U, no de
  línea.
- Los inactivos abandonan casi el doble que los activos (26,9 % frente a 14,3 %).
- Abandonan más quienes tienen saldo que quienes tienen saldo 0 (24,1 % frente a 14,0 %).
- Alemania duplica la tasa de Francia y España (32,5 % frente a 16,2 % y 16,7 %), y las mujeres abandonan más que los
  hombres (25,1 % frente a 16,4 %). Aun así, país y género no entran al modelo, por decisión ética.
- La antigüedad y la tarjeta casi no cambian la tasa (5,9 y 0,6 pp de diferencia).

### 3.2 Variables numéricas por tramos

Mismo formato de 3.1; los tramos están en `TRAMOS`.

In [ ]:
# Tramos de cada variable: (bordes, etiquetas). Cada tramo incluye su borde inferior y excluye el superior.
TRAMOS = {
    'credit_score': ([350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 851],
                     ['350–399', '400–449', '450–499', '500–549', '550–599', '600–649', '650–699',
                      '700–749', '750–799', '800–850']),
    'age': ([18, 25, 35, 45, 55, 65, 75, 101],
            ['18–24', '25–34', '35–44', '45–54', '55–64', '65–74', '75+']),
    'balance': ([0, 0.01, 50_000, 100_000, 150_000, 200_000, 260_000],
                ['0', '0–50k', '50k–100k', '100k–150k', '150k–200k', '200k+']),
    'estimated_salary': ([0, 25_000, 50_000, 75_000, 100_000, 125_000, 150_000, 175_000, 200_001],
                         ['0–25k', '25k–50k', '50k–75k', '75k–100k', '100k–125k', '125k–150k',
                          '150k–175k', '175k–200k']),
}

tasas_tramos = {}
for col, (bordes, etiquetas) in TRAMOS.items():
    tramo = pd.cut(df[col], bins=bordes, labels=etiquetas, right=False)
    assert tramo.notna().sum() == df[col].notna().sum(), f'Hay valores de {col} fuera de los tramos'
    tasas_tramos[col] = tasa_churn(df.assign(tramo=tramo), 'tramo')

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, col in zip(axes.ravel(), NUM_MUCHOS_VALORES):
    barras_churn(ax, tasas_tramos[col], col, fuente=7)
    ax.set_xlabel('')
fig.legend(leyenda, ['se queda', 'abandona'], loc='upper right', ncol=2)
fig.suptitle(f'Churn por tramos: alto = clientes del tramo; encima, % que abandona y conteo '
             f'(promedio: {tasa_global:.1f}%)', fontweight='bold')
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f'{RUTA_FIGS}/03_numericas_por_churn.png', bbox_inches='tight'); plt.show()

medianas = df.groupby(OBJETIVO)[NUM_MUCHOS_VALORES + NUM_POCOS_VALORES].median().T
medianas.columns = ['mediana_se_queda', 'mediana_abandona']
medianas


**Qué vemos:**

- La edad es la variable numérica que más cambia la tasa. El abandono es del 7,9 % entre los 25 y 34 años, sube al
  47,8 % entre los 45 y 54 y al 49,7 % entre los 55 y 64, y vuelve a bajar después (18,2 % entre los 65 y 74). La
  relación no es lineal. La mediana de edad de quienes abandonan es 45 años, frente a 36 de quienes se quedan.
- En el saldo, el tramo de 100.000 a 150.000 abandona más (25,8 %) que el de saldo 0 (14,0 %).
- El puntaje y el salario casi no cambian la tasa, que ronda el 20 % en casi todos los tramos. Las medianas de quienes
  se quedan y de quienes abandonan son casi iguales. El tramo de puntaje 350–399 abandona completo, pero son solo 16
  clientes.

## 4. Correlación de Spearman

In [ ]:
cols_corr = NUM_MUCHOS_VALORES + NUM_POCOS_VALORES + BINARIAS + [OBJETIVO]
corr = df[cols_corr].corr(method='spearman')

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlación de Spearman', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/03_correlacion.png', bbox_inches='tight'); plt.show()

corr[OBJETIVO].drop(OBJETIVO).sort_values(key=abs, ascending=False).to_frame('correlacion_con_churn')


**Qué vemos:**

- La edad tiene la correlación más alta con el churn (0,32). Le siguen la actividad (−0,16), el número de productos
  (−0,12) y el saldo (0,11). El resto está cerca de 0.
- Entre variables, solo el saldo y el número de productos se relacionan (−0,32). No hay redundancia fuerte que obligue
  a quitar variables.
- Spearman solo mide relaciones monótonas. Por eso subestima el efecto de los productos (en forma de U) y de la edad
  (sube y luego baja), que se vieron en 3.1 y 3.2. Esto sugiere probar un modelo que capte relaciones no lineales.